In [343]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
import pandas as pd

In [344]:
df=pd.read_csv('../data/processed/match_data.csv')
df.head()


,match_id,batting_team,bowling_team,toss_winner,toss_decision,venue,city,season,match_won_by,player_of_match,batting_team_home,bowling_team_home,batting_team_won_toss,is_knockout,stage_clean,year,month,day
0,335982,Kolkata Knight Riders,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,M Chinnaswamy Stadium,Bangalore,2007/08,Kolkata Knight Riders,BB McCullum,0,1,0,0,League,2008,4,18
1,335983,Chennai Super Kings,Punjab Kings,Chennai Super Kings,bat,Punjab Cricket Association Stadium,Chandigarh,2007/08,Chennai Super Kings,MEK Hussey,0,0,1,0,League,2008,4,19
2,335984,Rajasthan Royals,Delhi Capitals,Rajasthan Royals,bat,Feroz Shah Kotla,Delhi,2007/08,Delhi Capitals,MF Maharoof,0,1,1,0,League,2008,4,19
3,335985,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Wankhede Stadium,Mumbai,2007/08,Royal Challengers Bengaluru,MV Boucher,1,0,1,0,League,2008,4,20
4,335987,Punjab Kings,Rajasthan Royals,Punjab Kings,bat,Sawai Mansingh Stadium,Jaipur,2007/08,Rajasthan Royals,SR Watson,0,1,1,0,League,2008,4,21


In [345]:
X=df.drop(['match_won_by','player_of_match','match_id'], axis=1)
y=df['match_won_by']
y.value_counts()

match_won_by
Mumbai Indians                 138
Chennai Super Kings            133
Kolkata Knight Riders          120
Royal Challengers Bengaluru    119
Punjab Kings                   108
Rajasthan Royals               106
Delhi Capitals                 104
Sunrisers Hyderabad             87
Gujarat Titans                  39
Lucknow Super Giants            32
Name: count, dtype: int64

In [346]:
X.nunique()

batting_team             10
bowling_team             10
toss_winner              10
toss_decision             2
venue                    37
city                     34
season                   19
batting_team_home         2
bowling_team_home         2
batting_team_won_toss     2
is_knockout               2
stage_clean               3
year                     19
month                     7
day                      31
dtype: int64

In [347]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 986 entries, 0 to 985
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   batting_team           986 non-null    object
 1   bowling_team           986 non-null    object
 2   toss_winner            986 non-null    object
 3   toss_decision          986 non-null    object
 4   venue                  986 non-null    object
 5   city                   986 non-null    object
 6   season                 986 non-null    object
 7   batting_team_home      986 non-null    int64 
 8   bowling_team_home      986 non-null    int64 
 9   batting_team_won_toss  986 non-null    int64 
 10  is_knockout            986 non-null    int64 
 11  stage_clean            986 non-null    object
 12  year                   986 non-null    int64 
 13  month                  986 non-null    int64 
 14  day                    986 non-null    int64 
dtypes: int64(7), object(8)


In [348]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [349]:
cols_one_hot = ['batting_team','bowling_team','toss_winner','toss_decision','venue','city','season','stage_clean']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cols_one_hot)
    ],
    remainder='passthrough'
)
X_train.dtypes

batting_team             object
bowling_team             object
toss_winner              object
toss_decision            object
venue                    object
city                     object
season                   object
batting_team_home         int64
bowling_team_home         int64
batting_team_won_toss     int64
is_knockout               int64
stage_clean              object
year                      int64
month                     int64
day                       int64
dtype: object

In [350]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [351]:
from sklearn.linear_model import LogisticRegression
le=LabelEncoder()
y_train=le.fit_transform(y_train)
y_test=le.transform(y_test)
model2=LogisticRegression(  C=1.0,
    penalty='l2',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)
model2.fit(X_train,y_train)

c:\Users\SRIHARSHITH\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000, random_state=42)

In [352]:
print("\t\t--------Logisitc Regression-------")
train_pred = model2.predict(X_train)
print("Train Accuracy:",
      accuracy_score(y_train, train_pred))
y_pred=model2.predict(X_test)
print("test_accuracy_score\n",accuracy_score(y_test,y_pred))
print("classification_report\n",classification_report(y_test,y_pred))
print("Confusion_matrix\n",confusion_matrix(y_test,y_pred))

		--------Logisitc Regression-------
Train Accuracy: 0.6586294416243654
test_accuracy_score
 0.5303030303030303
classification_report
               precision    recall  f1-score   support

           0       0.58      0.56      0.57        27
           1       0.48      0.60      0.53        20
           2       0.67      0.29      0.40         7
           3       0.55      0.60      0.57        20
           4       0.50      0.33      0.40         6
           5       0.64      0.64      0.64        33
           6       0.32      0.38      0.35        21
           7       0.54      0.52      0.53        25
           8       0.53      0.43      0.48        23
           9       0.59      0.62      0.61        16

    accuracy                           0.53       198
   macro avg       0.54      0.50      0.51       198
weighted avg       0.54      0.53      0.53       198

Confusion_matrix
 [[15  2  0  0  0  3  4  2  1  0]
 [ 1 12  0  0  1  3  1  0  1  1]
 [ 0  1  2  0  0  1  0

In [353]:
model=RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42
)
model.fit(X_train,y_train)

RandomForestClassifier(max_depth=10, min_samples_leaf=5, min_samples_split=10,
                       n_estimators=300, random_state=42)

In [361]:
from xgboost import XGBClassifier

model1 = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    random_state=42
)

model1.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [355]:
print("\t\t--------Random Forest-------")
train_pred = model.predict(X_train)
print("Train Accuracy:",
      accuracy_score(y_train, train_pred))
y_pred=model.predict(X_test)
print("test_accuracy_score\n",accuracy_score(y_test,y_pred))
print("classification_report\n",classification_report(y_test,y_pred))
print("Confusion_matrix\n",confusion_matrix(y_test,y_pred))

		--------Random Forest-------
Train Accuracy: 0.6154822335025381
test_accuracy_score
 0.5707070707070707
classification_report
               precision    recall  f1-score   support

           0       0.62      0.85      0.72        27
           1       0.53      0.45      0.49        20
           2       0.67      0.57      0.62         7
           3       0.50      0.65      0.57        20
           4       0.00      0.00      0.00         6
           5       0.64      0.64      0.64        33
           6       0.43      0.29      0.34        21
           7       0.56      0.60      0.58        25
           8       0.67      0.61      0.64        23
           9       0.47      0.50      0.48        16

    accuracy                           0.57       198
   macro avg       0.51      0.52      0.51       198
weighted avg       0.55      0.57      0.55       198

Confusion_matrix
 [[23  0  0  0  0  2  0  1  0  1]
 [ 2  9  1  0  0  3  2  1  1  1]
 [ 0  0  4  0  0  1  0  1  0

c:\Users\SRIHARSHITH\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\SRIHARSHITH\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\SRIHARSHITH\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [362]:
print("\t\t-------XGBoost-------")
train_pred = model1.predict(X_train)
print("Train Accuracy:",
      accuracy_score(y_train, train_pred))
y_pred=model1.predict(X_test)
print("test_accuracy_score\n",accuracy_score(y_test,y_pred))
print("classification_report\n",classification_report(y_test,y_pred))
print("Confusion_matrix\n",confusion_matrix(y_test,y_pred))

		-------XGBoost-------
Train Accuracy: 0.8071065989847716
test_accuracy_score
 0.47474747474747475
classification_report
               precision    recall  f1-score   support

           0       0.55      0.67      0.60        27
           1       0.33      0.30      0.32        20
           2       0.33      0.29      0.31         7
           3       0.39      0.45      0.42        20
           4       0.83      0.83      0.83         6
           5       0.57      0.61      0.59        33
           6       0.24      0.24      0.24        21
           7       0.48      0.40      0.43        25
           8       0.55      0.48      0.51        23
           9       0.53      0.50      0.52        16

    accuracy                           0.47       198
   macro avg       0.48      0.48      0.48       198
weighted avg       0.47      0.47      0.47       198

Confusion_matrix
 [[18  0  0  0  0  3  4  0  1  1]
 [ 3  6  1  0  0  3  2  2  2  1]
 [ 0  1  2  0  0  1  0  1  1  1]
 

In [363]:
import joblib
joblib.dump(model,"ipl_model.pkl")
joblib.dump(preprocessor,"preprocessor.pkl")
joblib.dump(le,"label-encoder.pkl")

['label-encoder.pkl']

In [370]:
new_match = pd.DataFrame({
    'batting_team': ['Chennai Super Kings'],
    'bowling_team': ['Mumbai Indians'],
    'toss_winner': ['Mumbai Indians'],
    'toss_decision': ['bowl'],
    'venue': ['Wankhede'],
    'city': ['Mumbai'],
    'season': ['2026'],
    'stage_clean': ['group stage'],
    'year': [2026],
    'month': [4],
    'day':[19],
    'batting_team_home': [0],
    'bowling_team_home': [1],
    'batting_team_won_toss': [0],
    'is_knockout': [0]
})

In [371]:
new_match_encoded = preprocessor.transform(new_match)
prediction = model.predict(new_match_encoded)
winner =le.inverse_transform(prediction)
print("Predicted Winner:", winner[0])

Predicted Winner: Mumbai Indians
